# 04 · Multilingual analysis

Retrieval per query language (Korean, English, Indonesian, Vietnamese, Chinese), per category, cross-language success (question language ≠ document language) and error cases. Each language has 20 questions per dataset - intervals are wide and per-language differences must be read with that in mind.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
frames = {}
for target in ("local", "production"):
    for ds in ("heldout_v0.2", "pilot_v0.1"):
        f = A.RESULTS / f"retrieval_{target}_{ds}.csv"
        if target == "production":   # erratum 2026-11-23: announcement ids corrected (scripts/rescore_production.py)
            f = A.RESULTS / f"retrieval_{target}_{ds}_rescored.csv"
        if f.exists():
            frames[(target, ds)] = A.load_retrieval(f.name)
avail = pd.DataFrame([(t, d, len(df), df["method"].nunique(), int((df["error"].fillna("") != "").sum()))
                      for (t, d), df in frames.items()], columns=["target", "dataset", "rows", "methods", "failed requests"])
avail

In [ ]:
manifest = {d["id"]: d for d in json.loads((A.SNAP / "corpus_manifest.json").read_text())["items"]}
bench = {ds: {r["question_id"]: r for r in json.loads((A.DATA / "benchmarks" / f"retrieval_{ds}.json").read_text())["records"]}
         for ds in ("heldout_v0.2", "pilot_v0.1")}
def doc_lang(ds, qid):
    return (manifest.get(bench[ds][qid]["primary_document_id"]) or {}).get("language")
for (t, d), df in frames.items():
    df["doc_language"] = df["question_id"].map(lambda q: doc_lang(d, q))
    df["cross_language"] = df["language"] != df["doc_language"]
main_key = ("production", "heldout_v0.2") if ("production", "heldout_v0.2") in frames else ("local", "heldout_v0.2")
print("main analysis:", main_key)

In [ ]:
by_lang = []
for (t, d), df in frames.items():
    s = A.summarize(df, ["method", "language"]); s.insert(0, "dataset", d); s.insert(0, "target", t); by_lang.append(s)
by_lang = pd.concat(by_lang, ignore_index=True)
by_lang.to_csv(A.RESULTS / "retrieval_by_language.csv", index=False)
pivot = by_lang[(by_lang.target == main_key[0]) & (by_lang.dataset == main_key[1])].pivot(index="language", columns="method", values=["r1", "rr"])
pivot.loc[A.LANGS].round(3)

In [ ]:
tab = by_lang[(by_lang.target == main_key[0]) & (by_lang.dataset == main_key[1])].set_index(["method", "language"])
methods = [m for m in ["B0", "E1", "E2", "E2S"] if m in set(tab.index.get_level_values(0))]
fig, axes = plt.subplots(2, 1, figsize=(11, 7.4))
for ax, metric, label in ((axes[0], "r1", "Recall@1"), (axes[1], "rr", "MRR")):
    A.grouped_bars(ax, tab.rename(index=A.LANG_NAMES, level=1), [A.LANG_NAMES[l] for l in A.LANGS], methods, metric)
    ax.set_ylabel(f"{label} (95% CI)")
    ax.set_title(f"{label} by query language · {main_key[0]} · held-out v0.2 (n = 20 questions per language)")
axes[0].legend(loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout(); print(A.save(fig, "04_recall_mrr_by_language"))

## Cross-language retrieval

A question is *cross-language* when its language differs from the original language of its primary expected document (most documents are English or Korean).

In [ ]:
xl = []
for (t, d), df in frames.items():
    s = A.summarize(df, ["method", "cross_language"], metrics=("r1", "rr")); s.insert(0, "dataset", d); s.insert(0, "target", t); xl.append(s)
xl = pd.concat(xl, ignore_index=True)
xl[["target", "dataset", "method", "cross_language", "n", "r1", "r1_lo", "r1_hi", "rr"]].round(3)

## Category-level performance (held-out, E2)

In [ ]:
df = frames[main_key]
cat = A.summarize(df[df.method == "E2"], ["category"], metrics=("r1", "r5", "rr"))
cat.sort_values("r1")[["category", "n", "r1", "r1_lo", "r1_hi", "r5", "rr"]].round(3)

## Error cases (held-out, E2: expected document not ranked first)

In [ ]:
titles = {k: v["title"][:45] for k, v in manifest.items()}
err = df[(df.method == "E2") & (df.r1 == 0)].copy()
err["question"] = err["question_id"].map(lambda q: bench[main_key[1]][q]["question"])
err["expected (primary)"] = err["question_id"].map(lambda q: titles.get(bench[main_key[1]][q]["primary_document_id"]))
err["retrieved #1"] = err["retrieved_documents"].map(lambda s: titles.get((json.loads(s) or [None])[0], "(nothing)"))
display(err[["question_id", "language", "rank", "question", "expected (primary)", "retrieved #1"]].sort_values("question_id"))
counts = err.groupby("language").size().reindex(A.LANGS, fill_value=0)
worst = counts.idxmax()
display(Markdown(f"""**Interpretation.** {len(err)} of {int((df.method == 'E2').sum())} held-out questions miss at rank 1 under
E2 ({', '.join(f'{A.LANG_NAMES[l]} {counts[l]}' for l in A.LANGS)}). With 20 questions per language a difference of
one question is 5 percentage points; per-language rankings should not be generalised without the reviewed, larger
dataset. Misses cluster on scenarios whose answer sits in one long multi-topic guide or in another-language document."""))